# #176 — Hai server vLLM trên một L4: `anser-v3-awq` (n8n) và `anser-v5-awq` (phần còn lại)

Phương án b của #91 (RUNBOOK §9.3), làm vì #174 (hai LoRA trên nền AWQ) trượt chất lượng: v3 rời 9/34 so với v3 gộp 33/34.
**Không train, không sinh dữ liệu, không gọi API ngoài.** Khoảng 40 đến 60 phút L4.

| Bước | Việc trong #176 | Ô |
|---|---|---|
| 1 | Kế hoạch VRAM, overhead đo ở #124 (1,64 GiB), `--tran-tong 0.88` | 2 |
| 2 | Bật hai server, **10 phút tải**, VRAM đỉnh và p95 | 3, 4 |
| 3 | n8n trên server A, bóc yêu cầu + diễn giải (+ agentic) trên server B, bằng `benchmark_v3` | 4, 5 |
| 4 | Ngữ cảnh tối thiểu mỗi server, từ **số token server vLLM đếm** cho từng lời gọi (`usage`), không đoán | 5 |

Ô 6 lưu mọi thứ lên Drive và in sẵn bình luận để dán vào #176.

**Trước khi chạy:** runtime **L4**. Trên Drive có `generated/` (đủ bốn tập `eval_*.jsonl`), `anser-v3-awq/`, `anser-v5-awq/`.
Sửa đường dẫn ở ô 1.2 nếu Drive của bạn đặt khác. Tập đo là câu hư cấu, không đưa dữ liệu khách vào phiên này (R2).
Colab chỉ để đo, không phục vụ traffic thật (AGENTS §3.1).

## 1. Chuẩn bị

In [ ]:
# 1.1 — GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 1.2 — Drive + mã nguồn + kiểm tập đo
import datetime
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

# ==== SỬA NẾU DRIVE CỦA BẠN KHÁC ====
GOC     = Path("/content/drive/MyDrive/ANSER_AI_Logistics")
GEN_DIR = GOC / "generated"
MODEL   = {"anser-v3": GOC / "anser-v3-awq",     # server A: n8n
           "anser-v5": GOC / "anser-v5-awq"}     # server B: bóc yêu cầu, diễn giải, agentic
# ====================================

REPO   = "https://github.com/ANSER-Logistics/ANSER_AI_Logistics.git"
# Đếm token từng lời gọi (bước 4) vào nhánh tính năng qua PR của #176. Chưa gộp thì để nhánh của PR;
# gộp rồi thì đổi thành "feat/workflow-format-and-deterministic-core".
BRANCH = "feat/176-ngu-canh-can-thiet"
WORK   = Path("/content/ANSER_AI")

drive.mount("/content/drive")
if WORK.exists():
    shutil.rmtree(WORK)
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORK)],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit(f"Không clone được:\n{r.stderr}")
os.chdir(WORK)
SHA = subprocess.run(["git", "log", "-1", "--format=%h"], capture_output=True, text=True).stdout.strip()
print("commit:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

thieu = [f for f in ("eval_n8n", "eval_extraction", "eval_narration", "eval_agent")
         if not (GEN_DIR / f"{f}.jsonl").exists()]
assert not thieu, f"thiếu tập đo trong {GEN_DIR}: {thieu}"
for ten, d in MODEL.items():
    assert (d / "config.json").exists(), f"{ten}: không thấy {d}/config.json"
os.environ["ANSER_GENERATED_DIR"] = str(GEN_DIR)

OUT = GOC / f"do_176_{datetime.date.today():%Y%m%d}_{SHA}"
OUT.mkdir(parents=True, exist_ok=True)
KQ = Path("/content/kq176")
KQ.mkdir(exist_ok=True)
print("kết quả sẽ lưu ở", OUT)

In [ ]:
# 1.3 — Thư viện: đúng bản production (vLLM 0.11.2, PR #125), cài theo requirements-gpu.txt.
# Chạy NGAY SAU 1.2 trong phiên mới: transformers không được import trước khi tắt TensorFlow.
import os

os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
!pip install -q "vllm==0.11.2" -r requirements-gpu.txt
!pip uninstall -y -q tensorflow tf-keras || true

import transformers
import vllm
from transformers.utils import is_tf_available

assert not is_tf_available(), "transformers vẫn thấy TensorFlow: Runtime > Restart, chạy lại 1.2 rồi 1.3"
assert vllm.__version__.startswith("0.11."), f"vLLM {vllm.__version__}: phép đo phải trên 0.11.x như production"
print("vLLM", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
# 1.4 — Kiểm hai model AWQ + chép tokenizer chuẩn từ model gốc (cùng cách các phiên trước: ô 1.4 của
# do_lai_br33_colab.ipynb; tokenizer do transformers v5 ghi lúc quantize đọc bằng 4.x thì nổ).
import json

from huggingface_hub import snapshot_download

FILES = ["tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt",
         "special_tokens_map.json", "added_tokens.json"]
goc = Path(snapshot_download("Qwen/Qwen3-8B", allow_patterns=FILES))
for ten, d in MODEL.items():
    cfg = json.loads((d / "config.json").read_text())
    q = cfg.get("quantization_config")
    assert q, f"{ten}: config.json thiếu quantization_config — vLLM sẽ nạp như fp16 rồi hết VRAM"
    idx = json.loads((d / "model.safetensors.index.json").read_text())
    thieu = [s for s in sorted(set(idx["weight_map"].values())) if not (d / s).exists()]
    assert not thieu, f"{ten}: thiếu shard {thieu}"
    for f in FILES:
        if (goc / f).exists():
            shutil.copy2(goc / f, d / f)
    if (d / "chat_template.jinja").exists():
        (d / "chat_template.jinja").rename(d / "chat_template.jinja.bak")
    print(f"✓ {ten}: {q.get('quant_method')} {q.get('bits')} bit — tokenizer đã chép từ Qwen/Qwen3-8B")

## 2. Kế hoạch VRAM (bước 1)

`ke_hoach_vram.py` đọc trọng số và KV cache từ chính file model, hỏi `nvidia-smi` dung lượng GPU. Overhead **1,64 GiB** là số đo cho text ở #124;
`--tran-tong 0.88` để tổng không chạm cổng VRAM 92% (mặc định 0,92 cũ mâu thuẫn với cổng đó).
Lệnh `vllm serve` ở ô 3 dựng bằng **chính hàm** `lenh_serve` của script, không chép tay.

In [ ]:
# 2.1 — In kế hoạch (để dán vào issue)
V3, V5 = MODEL["anser-v3"], MODEL["anser-v5"]
!python offline_training/ke_hoach_vram.py \
    --model {V3} --ten anser-v3 \
    --model {V5} --ten anser-v5 \
    --overhead-gib 1.64 --tran-tong 0.88 2>&1 | tee {KQ}/ke_hoach.txt

In [ ]:
# 2.2 — Cùng kế hoạch, dạng đối tượng, để ô 3 bật server
import shlex
import sys

sys.path.insert(0, str(WORK))
from offline_training.ke_hoach_vram import doc_dac_ta, dung_luong_gpu_gib, lap_ke_hoach, lenh_serve

TEN = list(MODEL)
KH = lap_ke_hoach([doc_dac_ta(MODEL[t]) for t in TEN], dung_luong_gpu_gib(),
                  overhead_gib=1.64, tran_tong=0.88, ten=TEN)
LENH = {s.ten: lenh_serve(s, KH.max_model_len) for s in KH.servers}
CONG = {s.ten: s.cong for s in KH.servers}
for t in TEN:
    print(LENH[t])

## 3. Bật hai server, LẦN LƯỢT (bước 2)

Đợi server trước lên `/health` rồi mới bật server sau, kẻo hai bên cùng đo bộ nhớ trống. Log từng server ở `/content/kq176/server_*.log`.
Server chết lúc khởi động thì ô in 40 dòng cuối log và dừng: đó là kết quả hợp lệ, ghi nguyên văn vào issue.

In [ ]:
# 3.1
import re
import time
import urllib.request

PROC = {}


def doi_len(ten, gio=900):
    url = f"http://127.0.0.1:{CONG[ten]}/health"
    t0 = time.time()
    while time.time() - t0 < gio:
        if PROC[ten].poll() is not None:
            print(*(KQ / f"server_{ten}.log").read_text(errors="replace").splitlines()[-40:], sep="\n")
            raise SystemExit(f"{ten} CHẾT lúc khởi động (mã {PROC[ten].returncode}) — log ở trên")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return time.time() - t0
        except OSError:
            pass
        time.sleep(5)
    raise SystemExit(f"{ten} chưa lên sau {gio}s")


KHOI_DONG = {}
for ten in TEN:
    log = open(KQ / f"server_{ten}.log", "w")
    PROC[ten] = subprocess.Popen(shlex.split(LENH[ten]), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    KHOI_DONG[ten] = doi_len(ten)
    print(f"✓ {ten} lên sau {KHOI_DONG[ten]:.0f}s, cổng {CONG[ten]}")

KV = {}
for ten in TEN:
    s = (KQ / f"server_{ten}.log").read_text(errors="replace")
    kv = re.search(r"GPU KV cache size:\s*([\d,]+)\s*tokens", s)
    ss = re.search(r"Maximum concurrency for ([\d,]+) tokens per request:\s*([\d.]+)x", s)
    KV[ten] = {"kv_token": kv.group(1) if kv else None, "dong_thoi": ss.group(2) if ss else None}
    print(ten, KV[ten])
!nvidia-smi --query-compute-apps=pid,used_memory --format=csv | tee {KQ}/vram_sau_khoi_dong.txt
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv | tee -a {KQ}/vram_sau_khoi_dong.txt

## 4. Tải 10 phút: hai benchmark song song, lặp tới đủ giờ (bước 2 và 3)

Mỗi vòng chạy **cùng lúc**: server A đo n8n (kèm một lần thử lại như `/chat`), server B đo bóc yêu cầu, diễn giải và agentic.
Mỗi benchmark gửi 4 lời gọi đồng thời (`BENCH_SONG_SONG`, mặc định), nên GPU luôn phục vụ tới 8 chuỗi một lúc.
Vòng mới chỉ bắt đầu khi chưa đủ 10 phút. `nvidia-smi` ghi bộ nhớ mỗi 500 ms suốt thời gian đó.

**Vòng 1 là bảng điểm** (bước 3). Các vòng sau cho tải và thêm mẫu độ trễ; temperature 0 nên điểm n8n không nên đổi giữa các vòng.

In [ ]:
# 4.1
GIO_TAI = 600
mau_vram = subprocess.Popen(
    ["nvidia-smi", "--query-gpu=timestamp,memory.used,memory.total,utilization.gpu",
     "--format=csv,noheader,nounits", "-lms", "500", "-f", str(KQ / "vram.csv")])


def lenh_bench(ten, vong):
    url = f"openai:http://127.0.0.1:{CONG[ten]}/v1#{ten}"
    if ten == "anser-v3":
        nhanh = ["--skip", "extraction", "narration", "agent", "--thu-lai-n8n"]
    else:
        nhanh = ["--skip", "n8n"]
    return [sys.executable, "offline_training/benchmark_v3.py", "--model", url, "--no-gate", *nhanh,
            "--json", str(KQ / f"{ten}_v{vong}.json"), "--tho", str(KQ / f"{ten}_v{vong}_tho.jsonl")]


t0 = time.time()
vong = 0
while time.time() - t0 < GIO_TAI:
    vong += 1
    chay = {}
    for ten in TEN:
        f = open(KQ / f"{ten}_v{vong}.log", "w")
        chay[ten] = subprocess.Popen(lenh_bench(ten, vong), stdout=f, stderr=subprocess.STDOUT)
    for ten, p in chay.items():
        p.wait()
        print(f"vòng {vong} {ten}: mã {p.returncode} (log {ten}_v{vong}.log), {time.time() - t0:.0f}s")
    for ten in TEN:
        assert PROC[ten].poll() is None, f"server {ten} CHẾT trong lúc tải — xem server_{ten}.log"
TONG_GIAY = time.time() - t0
mau_vram.terminate()
mau_vram.wait()
print(f"xong {vong} vòng trong {TONG_GIAY:.0f}s")

In [ ]:
# 4.2 — Báo cáo vòng 1 (nguyên văn, để đọc ca hỏng)
for ten in TEN:
    print("=" * 30, ten, "vòng 1", "=" * 30)
    print((KQ / f"{ten}_v1.log").read_text(errors="replace")[-6000:])

## 5. Bảng: VRAM đỉnh, điểm, p95, ngữ cảnh tối thiểu (bước 2, 3, 4)

In [ ]:
# 5.1
import csv

dong = [r for r in csv.reader(open(KQ / "vram.csv")) if len(r) >= 3]
dung = [float(r[1]) for r in dong]
tong = float(dong[0][2])
VRAM = {"dinh_mib": max(dung), "tong_mib": tong, "ty_le": max(dung) / tong, "n_mau": len(dung)}
print(f"VRAM đỉnh {VRAM['dinh_mib']:.0f}/{tong:.0f} MiB = {VRAM['ty_le'] * 100:.1f}%"
      f"  ({'DƯỚI' if VRAM['ty_le'] < 0.92 else 'CHẠM'} cổng 92%; {len(dung)} mẫu)")

VONG = {ten: [json.loads(p.read_text()) for p in sorted(KQ.glob(f"{ten}_v*.json"),
                                                         key=lambda p: int(p.stem.split("_v")[1]))]
        for ten in TEN}
v1a = VONG["anser-v3"][0]["sections"]
v1b = VONG["anser-v5"][0]["sections"]


def ty(k, n):
    return f"{k}/{n}" if k is not None else "—"


DIEM = []
n8 = v1a.get("n8n", {})
DIEM.append(("A anser-v3", "n8n hợp lệ (lần đầu)", ty(n8.get("n_valid"), n8.get("n"))))
if "thu_lai" in n8:
    DIEM.append(("A anser-v3", "n8n sau một lần thử lại", ty(n8["thu_lai"]["n_valid"], n8["thu_lai"]["n"])))
ex = v1b.get("extraction", {})
DIEM.append(("B anser-v5", "bóc yêu cầu: sẵn sàng báo giá", ty(ex.get("n_ready"), ex.get("n"))))
na = v1b.get("narration", {})
DIEM.append(("B anser-v5", "diễn giải đạt", ty(na.get("n_pass"), na.get("n"))))
ag = v1b.get("agentic", {})
for k, nhan in (("ask_back_rate", "agentic biết hỏi lại"), ("call_rate", "agentic chịu gọi tool"),
                ("arg_fill_rate", "agentic điền tham số")):
    if ag.get(k) is not None:
        DIEM.append(("B anser-v5", nhan, f"{ag[k] * 100:.1f}%"))

# Token và độ trễ: gom mọi vòng. ctx cần = max qua các vòng (tất định với cùng prompt, nhưng thử lại n8n
# phụ thuộc đầu ra hỏng nên có thể khác). p95 lấy vòng XẤU NHẤT, không trung bình các p95.
NHANH = [("anser-v3", "n8n", lambda s: s.get("n8n", {}).get("token")),
         ("anser-v3", "n8n thử lại", lambda s: s.get("n8n", {}).get("thu_lai", {}).get("token")),
         ("anser-v5", "extraction", lambda s: s.get("extraction", {}).get("token")),
         ("anser-v5", "narration", lambda s: s.get("narration", {}).get("token")),
         ("anser-v5", "agentic", lambda s: s.get("agentic", {}).get("token"))]
TOKEN = []
for ten, nhan, lay in NHANH:
    tts = [t for v in VONG[ten] if (t := lay(v["sections"]))]
    if not tts:
        TOKEN.append((ten, nhan, None))
        continue
    TOKEN.append((ten, nhan, {
        "prompt_max": max(t["prompt_max"] for t in tts), "max_tokens": max(t["max_tokens"] for t in tts),
        "da_dung_max": max(t["da_dung_max"] for t in tts), "ctx_can": max(t["ctx_can"] for t in tts),
        "giay_p50": max(t["giay_p50"] for t in tts), "giay_p95": max(t["giay_p95"] for t in tts),
        "n_goi": sum(t["n_goi"] for t in tts), "n_thieu_usage": sum(t["n_thieu_usage"] for t in tts)}))

CTX_CAN = {ten: max([t["ctx_can"] for x, _, t in TOKEN if x == ten and t] or [0]) for ten in TEN}

print("\n| Server | Chỉ số (vòng 1) | Kết quả |\n|---|---|---|")
for s, k, v in DIEM:
    print(f"| {s} | {k} | {v} |")
print("\n| Server | Nhánh | lời gọi | prompt max | trần sinh | đã dùng max | **ctx cần** | giây p50 | giây p95 |")
print("|---|---|---|---|---|---|---|---|---|")
for ten, nhan, t in TOKEN:
    if t is None:
        print(f"| {ten} | {nhan} | — | không đo được (không có lời gọi hoặc thiếu usage) | | | | | |")
    else:
        print(f"| {ten} | {nhan} | {t['n_goi']} | {t['prompt_max']} | {t['max_tokens']} | {t['da_dung_max']}"
              f" | **{t['ctx_can']}** | {t['giay_p50']} | {t['giay_p95']} |")
for ten in TEN:
    print(f"\n{ten}: --max-model-len đang chạy {KH.max_model_len}, cần tối thiểu {CTX_CAN[ten]}"
          f" ({'đủ' if CTX_CAN[ten] <= KH.max_model_len else 'THIẾU'}) trên bộ đề")

**Đọc bảng cho đúng:** "ctx cần" = `max(prompt + max_tokens)` trên mọi lời gọi của bộ đề, vì vLLM từ chối theo trần xin chứ không theo số đã sinh.
Đó là **sàn đo trên bộ đề**, không phải trần của production: `/chat` thật có thêm lịch sử hội thoại (tối đa 6 tin, mỗi tin tối đa 4.000 ký tự)
và kết quả tool trong vòng agentic, nên prompt thật có thể dài hơn. Phần dư giữa "ctx cần" và `--max-model-len` là chỗ cho những thứ đó.

## 6. Tắt server, lưu lên Drive, bình luận để dán vào #176

In [ ]:
# 6.1
import signal

for ten, p in PROC.items():
    if p.poll() is None:
        os.killpg(p.pid, signal.SIGTERM)
for p in PROC.values():
    try:
        p.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(p.pid, signal.SIGKILL)

json.dump({"commit": SHA, "vllm": vllm.__version__, "ke_hoach": {"max_model_len": KH.max_model_len,
           "servers": {s.ten: {"gpu_memory_utilization": round(s.phan_gpu, 2), "nhu_cau_gib": round(s.nhu_cau_gib, 2)}
                       for s in KH.servers}},
           "khoi_dong_giay": KHOI_DONG, "kv": KV, "vram": VRAM, "tai_giay": TONG_GIAY, "so_vong": vong,
           "diem_vong_1": DIEM, "token": TOKEN, "ctx_can": CTX_CAN},
          open(KQ / "tong_hop_176.json", "w"), ensure_ascii=False, indent=1)
for f in KQ.iterdir():
    shutil.copy2(f, OUT)
print("Đã lưu", len(list(OUT.iterdir())), "file vào", OUT)

print("\n" + "=" * 70 + "\nDÁN VÀO #176\n" + "=" * 70)
print(f"Phiên Colab L4, commit `{SHA}`, vLLM {vllm.__version__}. Kết quả: `MyDrive/ANSER_AI_Logistics/{OUT.name}/`.\n")
print("**Lệnh khởi động đã chạy** (lần lượt, đợi `/health`):\n```")
for t in TEN:
    print(LENH[t])
print("```")
for t in TEN:
    print(f"- {t}: lên sau {KHOI_DONG[t]:.0f}s; KV cache {KV[t]['kv_token']} token; đồng thời tối đa {KV[t]['dong_thoi']}x")
print(f"\n**VRAM đỉnh** trong {TONG_GIAY / 60:.1f} phút tải ({vong} vòng, hai benchmark song song, 4 lời gọi đồng thời mỗi bên):"
      f" {VRAM['dinh_mib']:.0f}/{VRAM['tong_mib']:.0f} MiB = **{VRAM['ty_le'] * 100:.1f}%**"
      f" ({'dưới' if VRAM['ty_le'] < 0.92 else 'CHẠM'} cổng 92%).\n")
print("**Bảng điểm (vòng 1)**\n\n| Server | Chỉ số | Kết quả |\n|---|---|---|")
for s, k, v in DIEM:
    print(f"| {s} | {k} | {v} |")
print("\n**Ngữ cảnh và độ trễ** (số token do server vLLM đếm; giây phía client, vòng xấu nhất)\n")
print("| Server | Nhánh | prompt max | trần sinh | ctx cần | giây p50 | giây p95 |\n|---|---|---|---|---|---|---|")
for ten, nhan, t in TOKEN:
    if t:
        print(f"| {ten} | {nhan} | {t['prompt_max']} | {t['max_tokens']} | {t['ctx_can']} | {t['giay_p50']} | {t['giay_p95']} |")
    else:
        print(f"| {ten} | {nhan} | không đo được | | | | |")
print("\n**Ngữ cảnh tối thiểu mỗi server** (sàn trên bộ đề; `/chat` thật thêm lịch sử và kết quả tool):")
for t in TEN:
    print(f"- {t}: {CTX_CAN[t]} token (đang chạy {KH.max_model_len})")